# Ex 3b: Variational Autoencoder on MNIST (PyTorch)

**Aim:** Train a VAE on MNIST with a 2-dimensional latent space and generate new digits.

## Imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

## Data

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

train_data = datasets.MNIST("./data", train=True, download=True, transform=transforms.ToTensor())
loader = DataLoader(train_data, batch_size=128, shuffle=True)

## Model

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 64)
        self.mean = nn.Linear(64, latent_dim)
        self.logvar = nn.Linear(64, latent_dim)
        self.fc3 = nn.Linear(latent_dim, 64)
        self.fc4 = nn.Linear(64, 256)
        self.fc5 = nn.Linear(256, 784)

    def encode(self, x):
        h = F.relu(self.fc1(x))
        h = F.relu(self.fc2(h))
        return self.mean(h), self.logvar(h)

    def reparameterize(self, mean, logvar):
        std = torch.exp(0.5 * logvar)
        return mean + std * torch.randn_like(std)

    def decode(self, z):
        h = F.relu(self.fc3(z))
        h = F.relu(self.fc4(h))
        return torch.sigmoid(self.fc5(h))

    def forward(self, x):
        mean, logvar = self.encode(x)
        z = self.reparameterize(mean, logvar)
        return self.decode(z), mean, logvar

## Loss

In [ ]:
def vae_loss(recon, x, mean, logvar):
    bce = F.binary_cross_entropy(recon, x, reduction="sum")
    kl = -0.5 * torch.sum(1 + logvar - mean.pow(2) - logvar.exp())
    return bce + kl, bce, kl

## Train

In [ ]:
model = VAE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 15
n_samples = len(loader.dataset)

for epoch in range(1, epochs + 1):
    total_loss, total_recon, total_kl = 0.0, 0.0, 0.0
    for x, _ in loader:
        x = x.view(-1, 784).to(device)
        optimizer.zero_grad()
        recon, mean, logvar = model(x)
        loss, recon_loss, kl_loss = vae_loss(recon, x, mean, logvar)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_loss.item()
    print(f"Epoch {epoch:2d}/{epochs} | loss: {total_loss / n_samples:.4f} | "
          f"recon: {total_recon / n_samples:.4f} | kl: {total_kl / n_samples:.4f}")

## Generate digits

In [ ]:
model.eval()
n = 15
size = 28
figure = np.zeros((size * n, size * n))
grid_x = np.linspace(-3, 3, n)
grid_y = np.linspace(-3, 3, n)[::-1]

with torch.no_grad():
    for i, yi in enumerate(grid_y):
        for j, xi in enumerate(grid_x):
            z = torch.tensor([[xi, yi]], dtype=torch.float32).to(device)
            digit = model.decode(z).cpu().numpy()[0].reshape(size, size)
            figure[i * size:(i + 1) * size, j * size:(j + 1) * size] = digit

plt.figure(figsize=(10, 10))
plt.imshow(figure, cmap="Greys_r")
plt.axis("off")
plt.title("Digits generated by sampling the VAE latent space")
plt.savefig("vae_generated_digits.png", dpi=150, bbox_inches="tight")
plt.show()

## Latent space scatter

In [ ]:
all_mean, all_labels = [], []
with torch.no_grad():
    for x, y in loader:
        mean, _ = model.encode(x.view(-1, 784).to(device))
        all_mean.append(mean.cpu().numpy())
        all_labels.append(y.numpy())
all_mean = np.concatenate(all_mean, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(all_mean[:, 0], all_mean[:, 1], c=all_labels, cmap="tab10", s=2)
plt.colorbar(scatter, label="Digit label")
plt.xlabel("z[0]")
plt.ylabel("z[1]")
plt.title("MNIST digits mapped into 2D latent space")
plt.savefig("vae_latent_space.png", dpi=150, bbox_inches="tight")
plt.show()